<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/05.%20Video%20Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Video Annotation**
> MARC - Multidisciplinary AI Research Centre | UOP


This notebook serves as the command interface for the **Wearable Biomechanics** project. It bridges the gap between raw Google Drive storage and Google Colab by using the **[OpenDatasetStore](https://github.com/janithcyapa/OpenDatasetStore)** utility.

The system maintains an automated, **JSON-indexed manifest**, ensuring that every subject and experiment is strictly organized with unique IDs and searchable metadata.

* **🛠️ Framework:** [OpenDatasetStore Library](https://github.com/janithcyapa/OpenDatasetStore)


## 📚 Initial Setup & Configuration


### 🔧 One-Time Setup Instructions

#### **Step 1: Get Access to the Shared Dataset Folder**
1. Ask your **admin** to share the Google Drive folder with you
2. You'll receive an email with a "Shared with you" notification
3. Open the shared folder in Google Drive
4. The folder should be named: **`Wearable-Biomechanics`**

#### **Step 2: Add Shortcut to Your Drive**
1. In Google Drive, locate the shared folder under **"Shared with me"**
2. **Right-click** on the folder named `Wearable-Biomechanics`
3. Select **"Add shortcut to Drive"** (or **"Organize → Add shortcut"**)
4. Navigate to your **`My Drive > Datasets`** folder
   - If `Datasets` folder doesn't exist, create it first
5. Click **"Add Shortcut"**

#### **Step 3: Verify the Setup**
- Go to **`My Drive > Datasets`** in your Google Drive
- You should see **`Wearable-Biomechanics`** with a shortcut icon ⤴️
- The full path will be: **`MyDrive/Datasets/Wearable-Biomechanics`**


### ⚠️ Important Notes
- 🔑 **Permissions**: You need at least **"Editor"** access to upload data
- 🔗 **Shortcuts**: Always use the shortcut in your Drive, not the original shared folder
- 📂 **Structure**: The system will auto-create `/index`, `/raw_data`, and `/processed_data` subfolders
- 🚫 **Don't delete** the shortcut or you'll lose access to shared data


### 📁 Folder Structure After Setup
```
My Drive/
└── Datasets/
    └── Wearable-Biomechanics/
        ├── index/
        │   ├── entities_subjects.json
        │   └── entries_experiments.json
        ├── raw_data/
        │   └── experiments/
        └── related_files/
        │    └── experiments/
        └── processed_data/
            └── experiments/
```


## **🛠️ Setup OpenDatasetStore**

In [1]:
%%capture
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

# @markdown ### Store Configuration
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}

store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local", # Use "local" since you are mounting Google Drive via Colab directly
    entity_id_format="sub_{num:04d}", # You can drop the `f""` string formatting here now
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)


In [5]:
# @title 🎬 Frame-by-Frame Video Player { display-mode: "form" }
import os, time, cv2, socket, functools, threading, subprocess
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
from IPython.display import display, HTML

entry_type      = "complete_trials"            # @param ["complete_trials", "fault_tolerant_trials"]
target_entry_id = "complete_trials_0001"       # @param {type:"string"}

# ── 1. Download synced video ───────────────────────────────────────────
video_dir = "/tmp/video_player"
os.makedirs(video_dir, exist_ok=True)
synced_video_path = store.download_related_file(
    entry_type=entry_type, entry_id=target_entry_id,
    label="synced_video", dest_folder=video_dir,
)
if not synced_video_path or not os.path.exists(synced_video_path):
    raise FileNotFoundError(f"❌ No synced_video for {target_entry_id}")

# ── 2. Always re-encode video on every run ─────────────────────────────
fast_proxy_path = os.path.join(video_dir, f"fast_proxy_{target_entry_id}.mp4")

print("⚡ Encoding video with current settings...")
cmd = [
    "ffmpeg", "-y", "-i", synced_video_path,
    "-vf", "scale=-2:1080",   # 1080p height (Delete these two items for 100% original resolution)
    "-c:v", "libx264",
    "-preset", "superfast",   # Keeps sharp edges without slowing down encoding
    "-tune", "fastdecode",    # Optimizes bitstream so the browser decodes frames faster when scrubbing
    "-crf", "20",             # Lower = sharper (18 = near-lossless, 20 = crisp HD, 24 = medium)
    "-g", "1",                # Every frame is a keyframe = instant frame stepping
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    "-an",
    fast_proxy_path
]
subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

video_filename = os.path.basename(fast_proxy_path)

# ── 3. Probe exact fps and frame count ────────────────────────────────
cap = cv2.VideoCapture(fast_proxy_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

# ── 4. Serve video file over HTTP ──────────────────────────────────────
def _free_port(start=8123):
    for p in range(start, start + 50):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try: s.bind(("", p)); return p
            except OSError: continue
    raise RuntimeError("no free port")

PORT = _free_port()
Handler = functools.partial(SimpleHTTPRequestHandler, directory=video_dir)
httpd = ThreadingHTTPServer(("", PORT), Handler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()

try:
    from google.colab import output
    output.no_vertical_scroll()
    base_url = (output.eval_js(f"google.colab.kernel.proxyPort({PORT})") or "").rstrip("/")
except Exception:
    base_url = f"http://localhost:{PORT}"

# Cache-buster (?t=...) forces the browser to load the newly encoded video every run
video_url = f"{base_url}/{video_filename}?t={int(time.time())}"

# ── 5. Build Compact, Balanced UI ──────────────────────────────────────
HTML_TEMPLATE = r"""
<div id="player_root" style="background:#18181b; color:#f4f4f5; padding:12px 14px; border-radius:10px; font-family:system-ui,sans-serif; width:100%; max-width:820px; box-sizing:border-box; border:1px solid #3f3f46; display:flex; flex-direction:column;">

  <!-- Header Bar -->
  <div style="display:flex; flex-wrap:wrap; justify-content:space-between; align-items:center; gap:8px; margin-bottom:8px;">
    <div style="font-size:14px; font-weight:bold; color:#38bdf8;">
      🎥 __ENTRY_ID__
    </div>

    <div style="display:flex; align-items:center; gap:6px; flex-wrap:wrap;">
      <span style="font-size:11px; color:#a1a1aa;">Size:</span>
      <button class="btn-sm" onclick="P.setSize(640, 320)">S</button>
      <button class="btn-sm" onclick="P.setSize(820, 420)">M</button>
      <button class="btn-sm" onclick="P.setSize(1080, 560)">L</button>
      <button class="btn-sm" onclick="P.toggleFullscreen()" title="Fullscreen (F)">⛶</button>
      <div id="readout" style="background:#27272a; padding:4px 10px; border-radius:5px; font-size:13px; color:#facc15; font-weight:bold; border:1px solid #52525b; font-variant-numeric:tabular-nums;">
        Frame: 0 / __TOTAL_M1__
      </div>
    </div>
  </div>

  <!-- Video Viewport -->
  <div id="video_box" style="background:#000; border-radius:6px; overflow:hidden; width:100%; height:480px; max-height:75vh; display:flex; justify-content:center; align-items:center;">
    <video id="video_player" src="__VIDEO_URL__" preload="auto" playsinline style="width:100%; height:100%; object-fit:contain; display:block; outline:none;"></video>
  </div>

  <!-- Timeline Scrubber -->
  <input type="range" id="scrubber" min="0" max="__TOTAL_M1__" value="0" step="1" style="width:100%; height:18px; margin-top:8px; cursor:pointer;" />

  <!-- Transport Controls -->
  <div style="display:flex; flex-wrap:wrap; gap:6px; margin-top:6px; align-items:center;">
    <button class="btn" onclick="P.step(-10)">⏪ -10</button>
    <button class="btn" onclick="P.step(-5)">-5</button>
    <button class="btn btn-acc" onclick="P.step(-1)">◀ -1</button>
    <button id="playBtn" class="btn" onclick="P.togglePlay()" style="background:#2563eb; color:white; font-weight:bold; min-width:80px;">▶ Play</button>
    <button class="btn btn-acc" onclick="P.step(1)">+1 ▶</button>
    <button class="btn" onclick="P.step(5)">+5</button>
    <button class="btn" onclick="P.step(10)">+10 ⏩</button>

    <select id="speed" onchange="P.setSpeed(this.value)" style="background:#27272a; color:white; padding:5px 8px; border-radius:5px; border:1px solid #52525b; cursor:pointer; font-size:12px;">
      <option value="0.1">0.1x</option>
      <option value="0.25">0.25x</option>
      <option value="0.5">0.5x</option>
      <option value="1" selected>1x</option>
      <option value="1.5">1.5x</option>
      <option value="2">2x</option>
    </select>

    <div style="display:flex; gap:4px; margin-left:auto;">
      <input id="jump" type="number" placeholder="Frame #" min="0" max="__TOTAL_M1__" style="width:80px; padding:5px 6px; background:#27272a; color:white; border:1px solid #52525b; border-radius:5px; font-size:12px;" />
      <button class="btn" onclick="P.jumpTo()">Go</button>
    </div>
  </div>

  <div style="margin-top:8px; font-size:11px; color:#a1a1aa;">
    💡 <kbd>Space</kbd> Play/Pause · <kbd>←</kbd>/<kbd>→</kbd> ±1 · <kbd>Shift</kbd>+<kbd>←</kbd>/<kbd>→</kbd> ±10 · <kbd>F</kbd> Fullscreen
  </div>
</div>

<style>
  .btn { padding: 5px 10px; background:#3f3f46; color:white; border:none; border-radius:5px; cursor:pointer; font-family:inherit; font-size:12px; font-weight:500; touch-action:manipulation; user-select:none; }
  .btn-sm { padding: 3px 8px; background:#27272a; color:#d4d4d8; border:1px solid #52525b; border-radius:4px; cursor:pointer; font-size:11px; }
  .btn:hover, .btn-sm:hover { filter: brightness(1.2); }
  .btn-acc { color:#facc15; font-weight:bold; }
  kbd { background:#27272a; padding:1px 5px; border-radius:3px; border:1px solid #52525b; font-size:10px; }

  #player_root:fullscreen { max-width:100vw !important; width:100vw; height:100vh; border-radius:0; border:none; padding:16px; }
  #player_root:fullscreen #video_box { height:auto !important; max-height:none !important; flex:1; }
</style>

<script>
(function(){
  const FPS      = __FPS__;
  const TOTAL_M1 = __TOTAL_M1__;

  const root     = document.getElementById("player_root");
  const vbox     = document.getElementById("video_box");
  const vid      = document.getElementById("video_player");
  const scrubber = document.getElementById("scrubber");
  const readout  = document.getElementById("readout");
  const playBtn  = document.getElementById("playBtn");
  let isSeeking  = false;

  function syncColabIframe() {
    if (window.google && google.colab && google.colab.output) {
      google.colab.output.setIframeHeight(document.body.scrollHeight + 16, true);
    }
  }

  function getFrame() {
    return Math.min(TOTAL_M1, Math.max(0, Math.round(vid.currentTime * FPS)));
  }

  function updateReadout(f) {
    readout.textContent = "Frame: " + f + " / " + TOTAL_M1;
    scrubber.value = f;
  }

  function setFrame(f) {
    f = Math.max(0, Math.min(TOTAL_M1, f | 0));
    vid.currentTime = Math.max(0, (f / FPS) + 0.0001);
    updateReadout(f);
  }

  function step(delta) {
    if (!vid.paused) vid.pause();
    setFrame(getFrame() + delta);
  }

  function togglePlay() {
    if (vid.paused) {
      if (getFrame() >= TOTAL_M1) setFrame(0);
      vid.play();
    } else {
      vid.pause();
    }
  }

  function toggleFullscreen() {
    if (!document.fullscreenElement) {
      root.requestFullscreen().catch(function(){
        if (vid.webkitEnterFullscreen) vid.webkitEnterFullscreen();
      });
    } else {
      document.exitFullscreen();
    }
  }

  vid.addEventListener("loadedmetadata", syncColabIframe);
  vid.addEventListener("play",  function(){ playBtn.textContent = "⏸ Pause"; });
  vid.addEventListener("pause", function(){ playBtn.textContent = "▶ Play"; updateReadout(getFrame()); });

  vid.addEventListener("timeupdate", function(){
    if (!isSeeking && !vid.paused) {
      updateReadout(getFrame());
    }
  });

  vid.addEventListener("ended", function(){
    playBtn.textContent = "▶ Play";
    updateReadout(TOTAL_M1);
  });

  scrubber.addEventListener("input", function(e){
    isSeeking = true;
    if (!vid.paused) vid.pause();
    setFrame(parseInt(e.target.value));
  });

  scrubber.addEventListener("change", function(){
    isSeeking = false;
  });

  document.addEventListener("keydown", function(e){
    var tag = document.activeElement.tagName;
    if (tag === "INPUT" || tag === "SELECT") {
      if (e.code === "Enter" && document.activeElement.id === "jump") P.jumpTo();
      return;
    }

    if (e.code === "Space"){
      e.preventDefault();
      togglePlay();
    } else if (e.code === "ArrowRight"){
      e.preventDefault();
      step(e.shiftKey ? 10 : 1);
    } else if (e.code === "ArrowLeft"){
      e.preventDefault();
      step(e.shiftKey ? -10 : -1);
    } else if (e.code === "KeyF"){
      e.preventDefault();
      toggleFullscreen();
    }
  });

  window.P = {
    step: step,
    togglePlay: togglePlay,
    toggleFullscreen: toggleFullscreen,
    setSpeed: function(v){ vid.playbackRate = parseFloat(v); },
    setSize: function(maxW, h){
      root.style.maxWidth = maxW + "px";
      vbox.style.height = h + "px";
      syncColabIframe();
    },
    jumpTo: function(){
      var v = parseInt(document.getElementById("jump").value);
      if (!isNaN(v)) step(v - getFrame());
    }
  };

  updateReadout(0);
  setTimeout(syncColabIframe, 150);
})();
</script>
"""

html = (HTML_TEMPLATE
        .replace("__VIDEO_URL__",  video_url)
        .replace("__FPS__",        repr(float(fps)))
        .replace("__TOTAL_M1__",   str(total_frames - 1))
        .replace("__ENTRY_ID__",   target_entry_id))
display(HTML(html))

📥 Downloaded to: /tmp/video_player/complete_trials_0001_sub_0004_synced_video.mp4
⚡ Encoding video with current settings...


<IPython.core.display.Javascript object>

In [5]:
# @title 📝 Per-Annotator Rep Phase Annotator { display-mode: "form" }
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# ── inherited from the video player cell ───────────────────────────
if "entry_type" not in globals() or "target_entry_id" not in globals():
    raise RuntimeError("⚠️ Run the video player cell first — it defines entry_type and target_entry_id.")
_entry_type      = entry_type
_target_entry_id = target_entry_id

N_ANNOTATORS = 10   # upper bound on the annotator ID picker
# ────────────────────────────────────────────────────────────────────

PHASES = [
    "start_rest", "setup", "first_pull", "second_pull", "clean",
    "middle_rest", "jerk_phase_one", "jerk_phase_two",
    "rest_up", "bring_down", "end_rest", "end_rep",
]

def new_rep_template():
    return {p: 0 for p in PHASES}

def tag_for(aid: int) -> str:
    return f"rep_segments_a{aid}"

state = {"annotator_id": None, "current_rep_idx": 0, "annotations": []}

# ── widgets ─────────────────────────────────────────────────────────
annotator_input = widgets.BoundedIntText(
    value=1, min=1, max=N_ANNOTATORS,
    description="Annotator ID:",
    layout=widgets.Layout(width="200px"),
)
rep_selector = widgets.Dropdown(
    options=[("Rep 1", 0)], value=0, description="Active Rep:",
    layout=widgets.Layout(width="180px"),
)
btn_add_rep    = widgets.Button(description="➕ Add Rep",         button_style="success", layout=widgets.Layout(width="120px"))
btn_del_rep    = widgets.Button(description="🗑️ Delete Rep",     button_style="danger",  layout=widgets.Layout(width="130px"))
btn_reload     = widgets.Button(description="🔄 Reload from store",                       layout=widgets.Layout(width="200px"))
btn_save_store = widgets.Button(description="💾 Save My Annotations", button_style="primary", layout=widgets.Layout(width="220px"))
comment_input  = widgets.Text(value="", placeholder="Optional comment", description="Comment:", layout=widgets.Layout(width="320px"))
status_label   = widgets.HTML(value="<i>Enter your Annotator ID to begin.</i>")

phase_inputs, phase_rows = {}, []
for phase in PHASES:
    int_box = widgets.IntText(
        value=0, description=phase,
        style={"description_width": "140px"},
        layout=widgets.Layout(width="340px"),
    )
    def _make_cb(p):
        def _cb(change):
            idx = state["current_rep_idx"]
            if 0 <= idx < len(state["annotations"]):
                state["annotations"][idx][p] = int(change["new"])
                refresh_preview()
        return _cb
    int_box.observe(_make_cb(phase), names="value")
    phase_inputs[phase] = int_box
    phase_rows.append(int_box)

json_preview = widgets.Textarea(
    value="", description="",
    layout=widgets.Layout(width="460px", height="450px"),
)

# ── helpers ─────────────────────────────────────────────────────────
def refresh_preview():
    json_preview.value = json.dumps(state["annotations"], indent=2)

def update_rep_selector():
    reps = state["annotations"]
    rep_selector.options = [(f"Rep {i+1}", i) for i in range(len(reps))]
    rep_selector.value = min(state["current_rep_idx"], len(reps) - 1) if reps else None

def load_rep_into_inputs(idx):
    reps = state["annotations"]
    if not reps:
        return
    idx = max(0, min(idx, len(reps) - 1))
    state["current_rep_idx"] = idx
    rep_selector.value = idx
    data = reps[idx]
    for p in PHASES:
        phase_inputs[p].value = int(data.get(p, 0))

def load_annotator(aid: int):
    state["annotator_id"]    = aid
    state["current_rep_idx"] = 0
    tag = tag_for(aid)
    try:
        df = store.get_entry_processed_data(
            entry_type=_entry_type, entry_id=_target_entry_id, tag=tag,
        )
    except Exception:
        df = None
    if df is not None and not df.empty:
        cols = [c for c in PHASES if c in df.columns]
        state["annotations"] = df[cols].to_dict(orient="records")
        status_label.value = (
            f"<span style='color:#38bdf8;'>📂 Loaded {len(state['annotations'])} rep(s) "
            f"for Annotator {aid} — tag <code>{tag}</code>.</span>"
        )
    else:
        state["annotations"] = [new_rep_template()]
        status_label.value = (
            f"<span style='color:#a1a1aa;'>📝 No existing data for Annotator {aid} — starting fresh "
            f"(will save to <code>{tag}</code>).</span>"
        )
    update_rep_selector()
    load_rep_into_inputs(0)
    refresh_preview()

# ── callbacks ───────────────────────────────────────────────────────
def on_annotator_change(change):
    if change["new"] is not None:
        load_annotator(int(change["new"]))

def on_rep_selected(change):
    if change["new"] is not None:
        state["current_rep_idx"] = change["new"]
        load_rep_into_inputs(change["new"])

def on_add_rep(_):
    if state["annotator_id"] is None:
        return
    state["annotations"].append(new_rep_template())
    new_idx = len(state["annotations"]) - 1
    update_rep_selector()
    load_rep_into_inputs(new_idx)
    status_label.value = (
        f"<span style='color:#38bdf8;'>➕ Added Rep {new_idx+1} "
        f"for Annotator {state['annotator_id']}.</span>"
    )

def on_del_rep(_):
    if state["annotator_id"] is None or len(state["annotations"]) <= 1:
        status_label.value = "<span style='color:#fbbf24;'>⚠️ At least 1 rep must remain.</span>"
        return
    removed = state["current_rep_idx"]
    state["annotations"].pop(removed)
    new_idx = max(0, removed - 1)
    update_rep_selector()
    load_rep_into_inputs(new_idx)
    status_label.value = f"<span style='color:#f87171;'>🗑️ Removed Rep {removed+1}.</span>"

def on_reload(_):
    if state["annotator_id"] is None:
        return
    load_annotator(state["annotator_id"])   # discards unsaved edits
    status_label.value = (
        f"<span style='color:#38bdf8;'>🔄 Reloaded Annotator {state['annotator_id']} from store "
        f"(unsaved edits discarded).</span>"
    )

def on_save(_):
    aid = state["annotator_id"]
    if aid is None:
        status_label.value = "<span style='color:#fbbf24;'>⚠️ Set an Annotator ID first.</span>"
        return

    # Adopt any manual edits from the JSON textarea
    try:
        parsed = json.loads(json_preview.value)
        if isinstance(parsed, list):
            state["annotations"] = parsed
    except Exception as e:
        status_label.value = f"<span style='color:#f87171;'>❌ Invalid JSON in preview: {e}</span>"
        return

    if not state["annotations"]:
        status_label.value = "<span style='color:#fbbf24;'>⚠️ Nothing to save.</span>"
        return

    rows = []
    for i, rep in enumerate(state["annotations"]):
        row = {"annotator_id": aid, "rep": i + 1}
        for p in PHASES:
            row[p] = int(rep.get(p, 0))
        rows.append(row)
    df  = pd.DataFrame(rows)[["annotator_id", "rep"] + PHASES]
    tag = tag_for(aid)

    # Replace-if-exists / add-if-new, ONLY for this annotator's tag
    try:
        existing = store.get_entry_processed_data(
            entry_type=_entry_type, entry_id=_target_entry_id, tag=tag,
        )
    except Exception:
        existing = None

    try:
        if existing is not None and not existing.empty:
            store.replace_processed_data(_entry_type, _target_entry_id, tag, df)
            verb = "Replaced"
        else:
            store.add_processed_data(
                entry_type=_entry_type, entry_id=_target_entry_id,
                df=df, tag=tag,
                comment=comment_input.value or f"annotator={aid}",
            )
            verb = "Saved"
        status_label.value = (
            f"<span style='color:#4ade80;'>✅ {verb} {len(df)} rep(s) for "
            f"Annotator {aid} → <code>{tag}</code> on <b>{_target_entry_id}</b>.</span>"
        )
    except Exception as e:
        status_label.value = f"<span style='color:#f87171;'>❌ Store error: {e}</span>"

annotator_input.observe(on_annotator_change, names="value")
rep_selector.observe(on_rep_selected, names="value")
btn_add_rep.on_click(on_add_rep)
btn_del_rep.on_click(on_del_rep)
btn_reload.on_click(on_reload)
btn_save_store.on_click(on_save)

# ── initial load ────────────────────────────────────────────────────
load_annotator(int(annotator_input.value))

# ── layout ──────────────────────────────────────────────────────────
header = widgets.HTML(
    f"<div style='padding:8px 12px; background:#18181b; color:#f4f4f5; "
    f"border-radius:6px; margin-bottom:8px; font-family:system-ui,sans-serif;'>"
    f"<b style='color:#38bdf8;'>🎬 {_target_entry_id}</b> "
    f"<span style='color:#71717a; font-size:12px;'>entry_type = "
    f"<code style='color:#fbbf24;'>{_entry_type}</code></span>"
    f"</div>"
)

top_bar    = widgets.HBox([annotator_input, btn_reload, comment_input, btn_save_store])
rep_bar    = widgets.HBox([rep_selector, btn_add_rep, btn_del_rep])
left_panel = widgets.VBox([widgets.HTML("<b>Phase Frame Indices (0-based):</b>")] + phase_rows)
right_panel= widgets.VBox([
    widgets.HTML("<b>Live JSON (your reps — editable before saving):</b>"),
    json_preview,
])
main_ui = widgets.VBox([
    header,
    status_label,
    top_bar,
    rep_bar,
    widgets.HBox([left_panel, right_panel], layout=widgets.Layout(gap="24px")),
])

display(main_ui)